In [ ]:
import os
import h5py
import cv2 as cv
import numpy as np
import pandas as pd
import concurrent.futures
import plotly.express as px
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
from image_preprocessing import (
    binarize_image,
    crop_image,
    resize_image,
    sift_shading,
    ImageDirPreprocessingPipeline,
)

# Configurations

In [ ]:
IMAGE_DIR = "processed/maithili_new_data/"

# Preprocessing
IMAGE_SHAPE = (128, 128)
GAUSSIAN_BLUR_KERNEL_SIZE = (5, 5)
SIFT_SHADING_GRID_SIZE = 8

# SIFT
FEATURES = 20
BINS = 150

# HOG
ORIENTATIONS = 9
PIXELS_PER_CELL = (8, 8)
CELLS_PER_BLOCK = (2, 2)

# Preprocessing

In [ ]:
PROCESSED_DIR = os.path.join('processed', 'resized_images')

preprocess_steps = [
    [binarize_image, GAUSSIAN_BLUR_KERNEL_SIZE],
    [crop_image],
    [resize_image, IMAGE_SHAPE],
]

if not os.path.exists(PROCESSED_DIR):
    for label in os.listdir(IMAGE_DIR):
        image_dir = os.path.join(IMAGE_DIR, label)
        out_dir = os.path.join(PROCESSED_DIR, label)
        preprocessor = ImageDirPreprocessingPipeline(image_dir, out_dir, "png", preprocess_steps)
        preprocessor.transform()

In [ ]:
def shuffle(df):
    return (
        df
        .sample(frac=1, random_state=420)
        .reset_index()
        .drop(["index"], axis=1)
    )

In [ ]:
def custom_train_test_split(df):
    df_train = shuffle(df.iloc[[i for i in range(df.shape[0]) if i%5 in [0,1,2]]])
    df_test = shuffle(df.iloc[[i for i in range(df.shape[0]) if i%5 in [3,4]]])
    return df_train.iloc[:,1:], df_test.iloc[:,1:], df_train.iloc[:,0], df_test.iloc[:,0]

# Machine Learning

## Histogram of Gradient

In [ ]:
def count_files_in_dir(directory):
    num_images = 0
    for _, _, files in os.walk(directory):
        num_images += len(files)
    return num_images

def hog_features_from_dir(file: h5py.File, dir: str, label: str):
    hog_feature_len = 8100
    num_images = count_files_in_dir(dir)
    hog_feature_shape = (num_images, hog_feature_len)

    dataset = file.create_dataset(label, shape=hog_feature_shape, dtype=np.float32)
    for i, image_name in enumerate(sorted(os.listdir(dir))):
        image_path = os.path.join(dir, image_name)
        image = cv.imread(image_path, cv.IMREAD_GRAYSCALE)
        feature_descriptors = hog(
            image,
            orientations=ORIENTATIONS,
            pixels_per_cell=PIXELS_PER_CELL,
            cells_per_block=CELLS_PER_BLOCK,
            visualize=False
        )
        dataset[i,:] = feature_descriptors

### HOG (Without Shade)

In [ ]:
from skimage.feature import hog

NO_SIFT_DIR = os.path.join('processed', 'resized_images')

image = cv.imread(os.path.join(NO_SIFT_DIR, "01", "001_1.png"), 0)
feature_descriptors, hog_image = hog(
    image,
    orientations=ORIENTATIONS,
    pixels_per_cell=PIXELS_PER_CELL,
    cells_per_block=CELLS_PER_BLOCK,
    visualize=True,
)

plt.subplot(121)
plt.imshow(image, cmap=plt.cm.gray)

plt.subplot(122)
plt.imshow(hog_image, cmap=plt.cm.binary)

plt.subplot(211)
plt.imshow(feature_descriptors, cmap=plt.cm.binary)

plt.show()

In [ ]:
print(f"Feature Descriptor Length: {len(feature_descriptors)}")

#### Extracting Hog Features

In [ ]:
dir = os.path.join("processed", "resized_images")
hog_dataset_name = os.path.join('datasets', 'hog_features_without_shade_128.h5')

num_images = count_files_in_dir(dir)
hog_feature_len = 8100
hog_features_shape = (num_images, hog_feature_len)

if not os.path.exists(hog_dataset_name):
    with h5py.File(hog_dataset_name, 'w') as f:
        for label in os.listdir(dir):
            image_dir = os.path.join(dir, label)
            hog_features_from_dir(f, image_dir, label)

#### Dimensionality Reduction

Finding out number of dimensions to reduce to

In [ ]:
from sklearn.decomposition import IncrementalPCA

labels_at_once = 13
n_components = 4000
inc_pca = IncrementalPCA(n_components=n_components)

with h5py.File(hog_dataset_name, 'r') as f:
    ls = []
    for i, data in enumerate(f.values(), 1):
        if i % labels_at_once == 0:
            inc_pca.partial_fit(np.concatenate(ls, axis=0))
            ls = []
        else:
            ls.append(data)

In [ ]:
variance = np.cumsum(inc_pca.explained_variance_ratio_)

In [ ]:
plt.plot(np.arange(len(variance)), variance)
plt.gca().get_xaxis().set_ticks(np.arange(0, 5001, 500))
plt.gca().get_yaxis().set_ticks(np.arange(0, 1.1, 0.1))
plt.xlabel("Dimensions")
plt.ylabel("Explained Variance")
plt.grid()
plt.show()

Reducing Dimensions

In [ ]:
from sklearn.decomposition import IncrementalPCA
dimensions = 4000
labels_at_once = 13

inc_pca = IncrementalPCA(n_components=dimensions)

with h5py.File(hog_dataset_name, 'r') as f:
    ls = []
    for i, data in enumerate(f.values(), 1):
        ls.append(data)
        if i % labels_at_once == 0:
            inc_pca.partial_fit(np.concatenate(ls, axis=0))
            ls = []

In [ ]:
hog_compressed_filename = os.path.join('datasets', 'hog_compressed_without_shade_4000dim.h5')
with h5py.File(hog_dataset_name, 'r') as original, h5py.File(hog_compressed_filename, 'w') as compressed:
    for label, value in original.items():
        dataset = compressed.create_dataset(label, shape=(value.shape[0], dimensions), dtype=np.float32)
        dataset[:] = inc_pca.transform(value)

#### DataFrame Generation (MinMaxScaler)

In [ ]:
hog_compressed_filename = os.path.join('datasets', 'hog_compressed_without_shade_4000dim.h5')
df_list = []
with h5py.File(hog_compressed_filename, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = MinMaxScaler(feature_range=(-1,1)).fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2, C=1)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
from sklearn.svm import SVC
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_hog_without_shade_minmax_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred =forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 400],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5,10],
   'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_hog_without_shade_minmax_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### DataFrame Generation (StandardScaler)

In [ ]:
df_list = []
with h5py.File(hog_compressed_filename, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = StandardScaler().fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
from sklearn.svm import SVC
svm_params = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_hog_without_shade_standard_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join("models", "forest_hog_without_shade_standard_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

### HOG (With Shade)

In [ ]:
from skimage.feature import hog

SIFT_DIR = os.path.join('processed', 'hog_shaded_128')

image = cv.imread(os.path.join(SIFT_DIR, "01", "001_1.png"), 0)
feature_descriptors, hog_image = hog(
    image,
    orientations=ORIENTATIONS,
    pixels_per_cell=PIXELS_PER_CELL,
    cells_per_block=CELLS_PER_BLOCK,
    visualize=True,
)

reshaped_descriptors = feature_descriptors[:8100].reshape(90,90)
plt.subplot(131)
plt.imshow(image, cmap=plt.cm.gray)

plt.subplot(132)
plt.imshow(hog_image, cmap=plt.cm.binary)

plt.subplot(133)
plt.imshow(reshaped_descriptors)
ticks =np.arange(0,reshaped_descriptors.shape[1],15)
plt.xticks(ticks)
plt.yticks(ticks)

plt.show()

In [ ]:
print(f"Feature Descriptor Length: {len(feature_descriptors)}")

#### Extracting Hog Features

In [ ]:
dir = SIFT_DIR
hog_dataset_name = os.path.join('datasets', 'hog_features_with_shade_128.h5')

num_images = count_files_in_dir(dir)
hog_feature_len = 8100
hog_features_shape = (num_images, hog_feature_len)

if not os.path.exists(hog_dataset_name):
    with h5py.File(hog_dataset_name, 'w') as f:
        for label in os.listdir(dir):
            image_dir = os.path.join(dir, label)
            hog_features_from_dir(f, image_dir, label)

#### Dimensionality Reduction

Finding out number of dimensions to reduce to

In [ ]:
from sklearn.decomposition import IncrementalPCA

labels_at_once = 13
n_components = 2000
inc_pca = IncrementalPCA(n_components=n_components)

with h5py.File(hog_dataset_name, 'r') as f:
    ls = []
    for i, data in enumerate(f.values(), 1):
        if i % labels_at_once == 0:
            inc_pca.partial_fit(np.concatenate(ls, axis=0))
            ls = []
        else:
            ls.append(data)

In [ ]:
variance = np.cumsum(inc_pca.explained_variance_ratio_)

In [ ]:
plt.plot(np.arange(len(variance)), variance)
plt.gca().get_xaxis().set_ticks(np.arange(0, 5001, 500))
plt.gca().get_yaxis().set_ticks(np.arange(0, 1.1, 0.1))
plt.xlabel("Dimensions")
plt.ylabel("Explained Variance")
plt.grid()
plt.show()

Reducing Dimensions

In [ ]:
from sklearn.decomposition import IncrementalPCA
dimensions=4000
labels_at_once = 13

inc_pca = IncrementalPCA(n_components=dimensions)

with h5py.File(hog_dataset_name, 'r') as f:
    ls = []
    for i, data in enumerate(f.values(), 1):
        ls.append(data)
        if i % labels_at_once == 0:
            inc_pca.partial_fit(np.concatenate(ls, axis=0))
            ls = []

In [ ]:
hog_compressed_filename = os.path.join('datasets', 'hog_compressed_shade_4000dim.h5')
with h5py.File(hog_dataset_name, 'r') as original, h5py.File(hog_compressed_filename, 'w') as compressed:
    for label, value in original.items():
        dataset = compressed.create_dataset(label, shape=(value.shape[0], dimensions), dtype=np.float32)
        dataset[:] = inc_pca.transform(value)

#### DataFrame Generation (MinMaxScaler)

In [ ]:
df_list = []
with h5py.File(hog_compressed_filename, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = MinMaxScaler(feature_range=(-1,1)).fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
from sklearn.svm import SVC
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear','rbf','poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_hog_with_shade_minmax_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join("models", "forest_hog_with_shade_minmax_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)


In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### DataFrame Generation (StandardScaler)

In [ ]:
df_list = []
with h5py.File(hog_compressed_filename, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = StandardScaler().fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred,))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
svm_params = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_hog_with_shade_standard_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_hog_with_shade_standard_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

## SIFT

In [ ]:
def sift_features_from_dir(file, dir, label):
    # HyperParameters
    BINS = 150
    FEATURES = 20
    SIFT_FEATURE_LEN = 128
    NUM_IMAGES = count_files_in_dir(dir)
    IMAGES = sorted(os.listdir(dir))
    SIFT_FEATURE_SHAPE = (NUM_IMAGES, BINS)

    total_sift_features = np.array([[0]*SIFT_FEATURE_LEN for _ in range(NUM_IMAGES*FEATURES)])
    final_features = file.create_dataset(label, shape=SIFT_FEATURE_SHAPE, dtype=np.float32)

    # Compute SIFT features
    sift = cv.SIFT_create()
    for i, file in enumerate(IMAGES):
        img_path = os.path.join(dir, file)
        img = cv.imread(img_path, 0)
        _, descriptors = sift.detectAndCompute(img, None)
        if descriptors is None:
            continue
        # random sampling with replacement
        desc_samples = descriptors[np.random.randint(descriptors.shape[0], size=FEATURES)]
        total_sift_features[(i*FEATURES):(i*FEATURES+FEATURES),] = desc_samples[:]

    # Create Bag of Words
    kmeans = KMeans(n_clusters=BINS, n_init=10)
    kmeans.fit(total_sift_features)
    for i, file in enumerate(IMAGES):
        img_path = os.path.join(dir, file)
        img = cv.imread(img_path, 0)
        _, descriptors = sift.detectAndCompute(img, None)
        if descriptors is None:
            continue
        idxs = kmeans.predict(descriptors.astype("float"))
        for idx in idxs:
            final_features[i,idx] += 1
    print(label, "done...")
    return final_features

### SIFT (With shading)

In [ ]:
SIFT_DIR = os.path.join('processed', 'sift_shaded_128')
SIFT_FEATURES_WITH_SIFT_FILENAME = os.path.join('datasets', 'sift_with_shade_128.h5')
if not os.path.exists(SIFT_FEATURES_WITH_SIFT_FILENAME):
    with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'w') as f:
        for label in os.listdir(SIFT_DIR):
            image_dir = os.path.join(SIFT_DIR, label)
            sift_features_from_dir(f, image_dir, label)

#### DataFrame Generation (MinMaxScaler)

In [ ]:
df_list = []
with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:,1:] = MinMaxScaler(feature_range=(-1, 1)).fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_sift_with_shade_minmax_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_sift_with_shade_minmax_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### DataFrame Generation (StandardScaler)

In [ ]:
df_list = []
with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = StandardScaler().fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_sift_with_shade_standard_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_sift_with_shade_standard_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train, y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

### SIFT (Without shading)

In [ ]:
SIFT_DIR = os.path.join('processed', 'resized_images')
SIFT_FEATURES_WITH_SIFT_FILENAME = os.path.join('datasets', 'sift_without_shade_128.h5')
if not os.path.exists(SIFT_FEATURES_WITH_SIFT_FILENAME):
    with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'w') as f:
        for label in os.listdir(dir):
            image_dir = os.path.join(dir, label)
            sift_features_from_dir(f, image_dir, label)

#### DataFrame Generation (MinMaxScaler)

In [ ]:
df_list = []
with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:,1:] = MinMaxScaler(feature_range=(-1, 1)).fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_sift_without_shade_minmax_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_sift_without_shade_minmax_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### DataFrame Generation (StandardScaler)

In [ ]:
df_list = []
with h5py.File(SIFT_FEATURES_WITH_SIFT_FILENAME, 'r') as f:
    for label, data in f.items():
        for row in data:
            df_list.append([label, *row])
df = pd.DataFrame(df_list)

In [ ]:
df.iloc[:, 1:] = StandardScaler().fit_transform(df.iloc[:,1:])

In [ ]:
X_train, X_test, y_train, y_test = custom_train_test_split(df)

In [ ]:
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

#### SVM

In [ ]:
from sklearn.svm import SVC
lin_svc = SVC(kernel='linear')
lin_svc.fit(X_train, y_train)

In [ ]:
lin_svc.score(X_test, y_test) * 100

In [ ]:
lin_svc.score(X_train, y_train)

In [ ]:
y_pred = lin_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (polynomial)

In [ ]:
poly_svc = SVC(kernel='poly', degree=2)
poly_svc.fit(X_train, y_train)

In [ ]:
poly_svc.score(X_test, y_test) * 100

In [ ]:
poly_svc.score(X_train, y_train)

In [ ]:
y_pred = poly_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### SVM (rbf)

In [ ]:
rbf_svc = SVC(kernel='rbf')
rbf_svc.fit(X_train, y_train)

In [ ]:
rbf_svc.score(X_test, y_test) * 100

In [ ]:
rbf_svc.score(X_train, y_train)

In [ ]:
y_pred = rbf_svc.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
px.imshow(confusion_matrix(y_test, y_pred))

#### Best SVM Model

In [ ]:
svm_params = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [1, 2, 3],
    'gamma': ['scale', 'auto']
}
best_svm = GridSearchCV(SVC(), svm_params, cv=5, n_jobs=-1)

In [ ]:
from joblib import dump, load
best_model_name = os.path.join('models', 'svm_sift_without_shade_standard_128.joblib')
if not os.path.exists(best_model_name):
    best_svm.fit(X_train, y_train)
    dump(best_svm, os.path.join(best_model_name))
best_svm = load(best_model_name)

In [ ]:
best_svm.best_params_

In [ ]:
best_svm.score(X_test, y_test) * 100

In [ ]:
best_svm.score(X_train, y_train)

In [ ]:
y_pred = best_svm.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest = RandomForestClassifier(random_state=420)
forest.fit(X_train, y_train)

In [ ]:
forest.score(X_test, y_test)

In [ ]:
forest.score(X_train, y_train)

In [ ]:
y_pred = forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best Forest

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
best_forest = GridSearchCV(RandomForestClassifier(n_jobs=-1), param_grid=param_grid, cv=5, n_jobs=-1)

In [ ]:
best_model_name = os.path.join("models", "forest_sift_without_shade_standard_128.joblib")
if not os.path.exists(best_model_name):
    best_forest.fit(X_train, y_train)
    dump(best_forest, os.path.join(best_model_name))
best_forest = load(best_model_name)

In [ ]:
best_forest.best_params_

In [ ]:
best_forest.score(X_test, y_test)

In [ ]:
best_forest.score(X_train, y_train)

In [ ]:
y_pred = best_forest.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
gnb.score(X_test, y_test)

In [ ]:
gnb.score(X_train, y_train)

In [ ]:
y_pred = gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)

#### Best NB

In [ ]:
param_grid = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
best_gnb = GridSearchCV(estimator=GaussianNB(), param_grid=param_grid, cv=5, n_jobs=-1)
best_gnb.fit(X_train , y_train)

In [ ]:
best_gnb.score(X_test, y_test)

In [ ]:
best_gnb.score(X_train, y_train)

In [ ]:
y_pred = best_gnb.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
px.imshow(cm)